# Ingestion

This notebook is to create an ETL (Extract, Transform, Load) Pipeline for the dashboard data.

- Request CSV from UK Government House Price Index (HPI) 
- Clean data for relevant information
  - Scotland only, 2005 onwards, correct data types for querying
- Save cleaned .csv file for live dashboard usage 

In [1]:
%pip install sqlalchemy streamlit datetime pandas numpy warnings plotly.express requests

  Using cached sqlalchemy-2.0.48-cp313-cp313-win_amd64.whl.metadata (9.8 kB)
  Using cached streamlit-1.55.0-py3-none-any.whl.metadata (9.8 kB)
  Using cached datetime-6.0-py3-none-any.whl.metadata (34 kB)
  Using cached pandas-3.0.1-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.4.2-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
Note: you may need to restart the kernel to use updated packages.


ERROR: Ignored the following versions that require a different python version: 0.55.2 Requires-Python <3.5; 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11; 1.26.0 Requires-Python >=3.9,<3.13; 1.26.1 Requires-Python >=3.9,<3.13
ERROR: Could not find a version that satisfies the requirement warnings (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for warnings


In [3]:
import pandas as pd
import numpy as np
from datetime import date
import os
import streamlit as st
import requests
import warnings
warnings.filterwarnings('ignore')


import plotly.express as px



In [ ]:

# CSV file: Full dataset
csv_url = "https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/UK-HPI-full-file-2025-12.csv?utm_medium=GOV.UK&utm_source=datadownload&utm_campaign=full_fil&utm_term=9.30_18_02_26"
full_data = "uk_house_price_index.csv"

# Send request
response = requests.get(csv_url)

# Check if the request was successful
if response.status_code == 200:
    with open(full_data, "wb") as file:
        file.write(response.content)
    print(f"Successfully downloaded: {full_data}")
else:
    print(f"Failed to download. Status code: {response.status_code}")

Successfully downloaded: uk_house_price_index.csv


In [31]:
# Load the CSV file into a DataFrame
df_data = pd.read_csv(full_data)
# Display the last few rows of the DataFrame
df_data.tail()

,Date,RegionName,AreaCode,AveragePrice,Index,IndexSA,1m%Change,12m%Change,AveragePriceSA,SalesVolume,...,NewPrice,NewIndex,New1m%Change,New12m%Change,NewSalesVolume,OldPrice,OldIndex,Old1m%Change,Old12m%Change,OldSalesVolume
149080,01/08/2025,Yorkshire and The Humber,E12000003,206107,107.6,105.8,0.9,2.0,202741.0,5471.0,...,302627.0,112.1,-0.9,7.4,19.0,202629.0,107.3,1.0,1.8,5452.0
149081,01/09/2025,Yorkshire and The Humber,E12000003,206581,107.8,106.9,0.2,3.9,204762.0,4745.0,...,306535.0,113.5,1.3,10.2,18.0,202921.0,107.5,0.1,3.5,4727.0
149082,01/10/2025,Yorkshire and The Humber,E12000003,206401,107.7,106.7,-0.1,3.2,204537.0,4961.0,...,313072.0,115.9,2.1,10.3,8.0,202381.0,107.2,-0.3,2.8,4953.0
149083,01/11/2025,Yorkshire and The Humber,E12000003,209467,109.3,108.0,1.5,3.8,206882.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
149084,01/12/2025,Yorkshire and The Humber,E12000003,208447,108.8,108.0,-0.5,3.3,206964.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
#FILTER for scottish data (all area code with scotland begin with "S")
scottish_index = df_data[df_data['AreaCode'].str.startswith('S')]['AreaCode'].unique()
df_scottish_data = df_data[df_data['AreaCode'].isin(scottish_index)]

In [33]:
#convert dates to correct format, 
#create ingestion date column
today = date.today().strftime('%d-%m-%Y')
df_scottish_data['Version_date'] = today
df_scottish_data['Version_date'] = pd.to_datetime(df_scottish_data['Version_date'])
df_scottish_data['Date'] = pd.to_datetime(df_scottish_data['Date'], format='%d/%m/%Y')

In [37]:
df_scottish_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9141 entries, 0 to 139142
Data columns (total 55 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Date                    9141 non-null   datetime64[ns]
 1   RegionName              9141 non-null   object        
 2   AreaCode                9141 non-null   object        
 3   AveragePrice            9141 non-null   int64         
 4   Index                   9141 non-null   float64       
 5   IndexSA                 264 non-null    float64       
 6   1m%Change               9106 non-null   float64       
 7   12m%Change              8745 non-null   float64       
 8   AveragePriceSA          264 non-null    float64       
 9   SalesVolume             8646 non-null   float64       
 10  DetachedPrice           8712 non-null   float64       
 11  DetachedIndex           8712 non-null   float64       
 12  Detached1m%Change       8679 non-null   float64    

In [38]:
#we are only interest in data from 2005 onwards

df_scottish2005 = df_scottish_data.loc[df_scottish_data['Date'] >= "2005-01-01"]

In [39]:
#filter for columns of interest
df_scottish2005_clean = df_scottish2005[['Date', 'RegionName', 'AveragePrice', 'SalesVolume', 'FlatPrice', 'SemiDetachedPrice', 'DetachedPrice', 'OldPrice', 'NewPrice', '12m%Change', 'Version_date']]

In [53]:
#change region into string dtype
df_scottish2005_clean['RegionName'] = df_scottish2005_clean["RegionName"].astype('category')

#rename '12m%change' column for readability
df_scottish2005_clean = df_scottish2005_clean.rename(columns = {'12m%Change' : 'Yearly%Change'})

In [54]:
df_scottish2005_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8316 entries, 12 to 139142
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Date               8316 non-null   datetime64[ns]
 1   RegionName         8316 non-null   category      
 2   AveragePrice       8316 non-null   int64         
 3   SalesVolume        8250 non-null   float64       
 4   FlatPrice          8316 non-null   float64       
 5   SemiDetachedPrice  8316 non-null   float64       
 6   DetachedPrice      8316 non-null   float64       
 7   OldPrice           8250 non-null   float64       
 8   NewPrice           8250 non-null   float64       
 9   Yearly%Change      8316 non-null   float64       
 10  Version_date       8316 non-null   datetime64[ns]
dtypes: category(1), datetime64[ns](2), float64(7), int64(1)
memory usage: 724.1 KB


In [56]:

#database store

from sqlalchemy import create_engine
engine = create_engine('sqlite:///housing.db')
df_scottish2005_clean.to_sql('scottish_housing_data_2005', con=engine, if_exists='replace', index=False)


8316

In [ ]:
#save as CSV
df_scottish2005_clean.to_csv('scot_housing_2005_clean.csv')